# Week 10 - Quora: Performance Measures, Train/Test Split & Basic EDA

**Topics (record sections d, e, f)**
* d. Performance measures
* e. Train/test split
* f. Basic statistics and first features (text lengths, word counts)

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/quora/train.csv")

## d. Performance measures
Let *positive* = "is a duplicate".

| Metric | Meaning | Why it matters here |
|---|---|---|
| **Accuracy** | share of correct predictions | misleading with imbalance |
| **Precision** | of pairs predicted duplicate, how many truly are | low precision = unrelated questions get merged (false positives) |
| **Recall** | of true duplicates, how many were found | low recall = duplicates slip through (false negatives) |
| **F1-score** | harmonic mean of precision & recall | single balanced number |
| **Log-loss** | $-\frac1n\sum[y\log p+(1-y)\log(1-p)]$ over predicted *probabilities* | penalises confident wrong answers; the Kaggle competition's metric (lower = better) |

A tiny worked example to see the metrics on real numbers:

In [2]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

y_true = np.array([1, 1, 1, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 0, 1, 0, 0, 0, 0])      # 2 TP, 1 FN, 1 FP, 4 TN
y_prob = np.array([.9, .8, .4, .6, .2, .1, .3, .2])

print("Accuracy :", accuracy_score(y_true, y_pred), "  (6 of 8 correct)")
print("Precision:", round(precision_score(y_true, y_pred), 3), "  = TP/(TP+FP) = 2/3")
print("Recall   :", round(recall_score(y_true, y_pred), 3), "  = TP/(TP+FN) = 2/3")
print("F1       :", round(f1_score(y_true, y_pred), 3))
print("Log-loss :", round(log_loss(y_true, y_prob), 3))

Accuracy : 0.75   (6 of 8 correct)
Precision: 0.667   = TP/(TP+FP) = 2/3
Recall   : 0.667   = TP/(TP+FN) = 2/3
F1       : 0.667
Log-loss : 0.384


## e. Clean the data and split
* The few missing question texts are replaced by an empty string so every row stays usable.
* **80/20 split**, `stratify=df["is_duplicate"]` keeps the duplicate ratio equal in both parts, `random_state=42` makes it repeatable.

In [3]:
df = df.dropna(subset=["is_duplicate"]).copy()
df["question1"] = df["question1"].fillna("")
df["question2"] = df["question2"].fillna("")
df["is_duplicate"] = df["is_duplicate"].astype(int)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["is_duplicate"])

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
print("\nTraining label distribution:")
print(train_df["is_duplicate"].value_counts(normalize=True))
print("\nTesting label distribution:")
print(test_df["is_duplicate"].value_counts(normalize=True))

Training shape: (323420, 6)
Testing shape: (80856, 6)

Training label distribution:
is_duplicate
0    0.63079
1    0.36921
Name: proportion, dtype: float64

Testing label distribution:
is_duplicate
0    0.630788
1    0.369212
Name: proportion, dtype: float64


## f. Basic statistics and simple features
Simple numeric features: the **length** (characters) of each question, their absolute **difference**, and the **word count**. Intuition: duplicates tend to have similar lengths.

In [4]:
print("Duplicate ratio:", df["is_duplicate"].mean())
print("\nDataset statistics:")
print(df[["is_duplicate"]].describe())

df["q1_length"] = df["question1"].str.len()
df["q2_length"] = df["question2"].str.len()
df["length_difference"] = (df["q1_length"] - df["q2_length"]).abs()
df["q1_word_count"] = df["question1"].str.split().str.len()
df["q2_word_count"] = df["question2"].str.split().str.len()

display(df[["question1", "question2", "is_duplicate", "q1_length", "q2_length", "length_difference"]].head())

Duplicate ratio: 0.36921063827682077

Dataset statistics:
        is_duplicate
count  404276.000000
mean        0.369211
std         0.482592
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000


,question1,question2,is_duplicate,q1_length,q2_length,length_difference
0,How is the life of a math student? Could you d...,Which level of prepration is enough for the ex...,0,75,55,20
1,How do I control my horny emotions?,How do you control your horniness?,1,35,34,1
2,What causes stool color to change to yellow?,What can cause stool to come out as little balls?,0,44,49,5
3,What can one do after MBBS?,What do i do after my MBBS ?,1,27,28,1
4,Where can I find a power outlet for my laptop ...,"Would a second airport in Sydney, Australia be...",0,67,121,54


In [5]:
print(df[["q1_length", "q2_length", "q1_word_count", "q2_word_count", "length_difference"]].describe().round(1))
print("\nMean length difference by class:")
print(df.groupby("is_duplicate")["length_difference"].mean().round(2))

       q1_length  q2_length  q1_word_count  q2_word_count  length_difference
count   404276.0   404276.0       404276.0       404276.0           404276.0
mean        59.6       60.2           10.9           11.2               20.2
std         30.0       33.9            5.4            6.3               25.6
min          1.0        1.0            1.0            1.0                0.0
25%         39.0       39.0            7.0            7.0                5.0
50%         52.0       51.0           10.0           10.0               12.0
75%         72.0       72.0           13.0           13.0               26.0
max        627.0     1175.0          125.0          237.0             1080.0

Mean length difference by class:
is_duplicate
0    24.33
1    13.16
Name: length_difference, dtype: float64


**Reading it:** duplicate pairs have a noticeably smaller length difference than non-duplicates - a weak but real signal that we will feed to the models.

## Summary
* Metrics chosen: accuracy, precision, recall, F1 and log-loss.
* Stratified 80/20 split (323k train / 81k test).
* First features: lengths, length difference, word counts.